In [1]:
import torch
import torch.nn as nn
import numpy as np
import math

In [2]:
def ScaledDotProductAttention(Q, K, V, mask=None):
    d_k = K.shape[-1]
    
    scores = Q @ K.transpose(-2, -1)  # (batch, seq_len, seq_len)
    scores = scores / math.sqrt(d_k)
    
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float('-inf'))
    
    scores = torch.softmax(scores, dim=-1)
    output = scores @ V
    
    return output

In [3]:
batch, seq_len, d_k = 2, 5, 64

Q = torch.randn(batch, seq_len, d_k)
K = torch.randn(batch, seq_len, d_k)
V = torch.randn(batch, seq_len, d_k)

out = ScaledDotProductAttention(Q, K, V)
print(out.shape)  # should be (2, 5, 64)

torch.Size([2, 5, 64])


In [4]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)
    
    def forward(self, Q, K, V, mask=None):
        batch = Q.shape[0]
        q_seq_len = Q.shape[1]
        k_seq_len = K.shape[1]
        
        Q = self.W_q(Q)
        K = self.W_k(K)
        V = self.W_v(V)
        
        Q = Q.view(batch, q_seq_len, self.num_heads, self.d_k).transpose(1, 2)
        K = K.view(batch, k_seq_len, self.num_heads, self.d_k).transpose(1, 2)
        V = V.view(batch, k_seq_len, self.num_heads, self.d_k).transpose(1, 2)
        
        output = ScaledDotProductAttention(Q, K, V, mask)
        
        output = output.transpose(1, 2).contiguous()
        output = output.view(batch, q_seq_len, self.num_heads * self.d_k)
        
        output = self.W_o(output)
        
        return output

In [5]:
batch, seq_len, d_model = 2, 5, 512
num_heads = 8

mha = MultiHeadAttention(d_model, num_heads)
Q = torch.randn(batch, seq_len, d_model)
K = torch.randn(batch, seq_len, d_model)
V = torch.randn(batch, seq_len, d_model)

out = mha(Q, K, V)
print(out.shape)  # should be (2, 5, 512)

torch.Size([2, 5, 512])


In [6]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_seq_len=512):
        super().__init__()
        # create a matrix of shape (max_seq_len, d_model)
        self.matrix = torch.zeros(max_seq_len,d_model)
        # fill even dims with sin, odd dims with cos

        positions = torch.arange(0, max_seq_len).unsqueeze(1)  
        dims = torch.arange(0, d_model, 2)                     
        denominator = torch.pow(10000, dims / d_model) 

        self.matrix[:,0::2] = torch.sin(positions/denominator) 
        self.matrix[:,1::2] = torch.cos(positions/denominator) 

        self.register_buffer('pe', self.matrix)      

        
    def forward(self, x):
        # add positional encoding to x and return
        seq_len = x.shape[1]
        x = x + self.pe[:seq_len,:]

        return x

In [7]:
batch, seq_len, d_model = 2, 5, 512

pe = PositionalEncoding(d_model)
x = torch.randn(batch, seq_len, d_model)

out = pe(x)
print(out.shape)  # should be (2, 5, 512)

torch.Size([2, 5, 512])


In [8]:
class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        # TODO: two linear layers
        self.l1 = nn.Linear(d_model,d_ff)
        self.relu = nn.ReLU()
        self.l2 = nn.Linear(d_ff,d_model)
        
    def forward(self, x):
        # TODO: linear → relu → linear
        
        x = self.l1(x)
        x = self.relu(x)
        x = self.l2(x)

        return x

In [9]:
batch, seq_len, d_model, d_ff = 2, 5, 512, 2048

ff = FeedForward(d_model, d_ff)
x = torch.randn(batch, seq_len, d_model)

out = ff(x)
print(out.shape)  # should be (2, 5, 512)

torch.Size([2, 5, 512])


In [10]:
class EncoderBlock(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        # you need:
        # - MultiHeadAttention
        self.mha = MultiHeadAttention(d_model,num_heads)
        # - FeedForward
        self.ffn = FeedForward(d_model,d_ff)
        # - two LayerNorms → nn.LayerNorm(d_model)
        self.layer_norm1 = nn.LayerNorm(d_model)
        self.layer_norm2 = nn.LayerNorm(d_model)
        # - dropout → nn.Dropout(dropout)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, x, mask=None):
        # attention → add & norm
        x = self.layer_norm1(x + self.mha(x, x, x, mask))
    
        # feedforward → add & norm
        x = self.layer_norm2(x + self.ffn(x))
        
        return x

In [11]:
batch, seq_len, d_model, num_heads, d_ff = 2, 5, 512, 8, 2048

encoder_block = EncoderBlock(d_model, num_heads, d_ff)
x = torch.randn(batch, seq_len, d_model)

out = encoder_block(x)
print(out.shape)  # should be (2, 5, 512)

torch.Size([2, 5, 512])


In [12]:
class DecoderBlock(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        # 2 MHAs, 3 LayerNorms, 1 FeedForward, 1 Dropout
        self.mha1 = MultiHeadAttention(d_model,num_heads)
        self.mha2 = MultiHeadAttention(d_model,num_heads)

        self.ffn = FeedForward(d_model,d_ff)

        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)
        self.ln3 = nn.LayerNorm(d_model)

        self.dropout = nn.Dropout(dropout)
        
    def forward(self, x, encoder_out, src_mask=None, tgt_mask=None):
        # masked self attention → add & norm
        x = self.ln1( x + self.mha1(x,x,x,tgt_mask))
        # cross attention → add & norm
        x = self.ln2( x + self.mha2(x,encoder_out,encoder_out,src_mask))
        # feedforward → add & norm
        x = self.ln3( x + self.ffn(x))

        return x

In [13]:
batch, seq_len, d_model, num_heads, d_ff = 2, 5, 512, 8, 2048

decoder_block = DecoderBlock(d_model, num_heads, d_ff)
x = torch.randn(batch, seq_len, d_model)
encoder_out = torch.randn(batch, seq_len, d_model)

out = decoder_block(x, encoder_out)
print(out.shape)  # should be (2, 5, 512)

torch.Size([2, 5, 512])


In [14]:
class Transformer(nn.Module):
    def __init__(self, src_vocab, tgt_vocab, d_model=512, 
                 num_heads=8, num_layers=6, d_ff=2048, dropout=0.1):
        super().__init__()
        
        # embeddings — one for source, one for target
        # hint: nn.Embedding(vocab_size, d_model)
        self.src_embedding = nn.Embedding(src_vocab,d_model)
        self.tgt_embedding = nn.Embedding(tgt_vocab,d_model)
        
        # positional encoding — one shared is fine
        self.pe = PositionalEncoding(d_model)
        
        # stack of N encoder blocks
        # hint: nn.ModuleList([EncoderBlock(...) for _ in range(num_layers)])
        self.encoder_layers = nn.ModuleList([EncoderBlock(d_model, num_heads, d_ff) for _ in range(num_layers)])
        
        # stack of N decoder blocks
        self.decoder_layers = nn.ModuleList([DecoderBlock(d_model, num_heads, d_ff) for _ in range(num_layers)])
        
        # final linear projection → vocab size
        # hint: maps d_model → tgt_vocab
        self.fc_out = nn.Linear(d_model,tgt_vocab)
        
        # dropout
        self.dropout = nn.Dropout(dropout)

    def forward(self, src, tgt, src_mask=None, tgt_mask=None):
        
        # step 1 — embed + positional encode src
        src = self.src_embedding(src) 
        src = self.pe(src)
        src = self.dropout(src)
        
        # step 2 — pass through encoder stack
        for layer in self.encoder_layers:
            src = layer(src,src_mask)
        
        # step 3 — embed + positional encode tgt
        tgt = self.tgt_embedding(tgt)
        tgt = self.pe(tgt)
        tgt = self.dropout(tgt)
        
        # step 4 — pass through decoder stack
        # remember decoder needs both tgt and encoder output (src)
        for layer in self.decoder_layers:
            tgt = layer(tgt,src,src_mask,tgt_mask)
        
        # step 5 — final linear projection
        output = self.fc_out(tgt)
        
        return output

In [15]:
src_vocab, tgt_vocab = 1000, 1000
model = Transformer(src_vocab, tgt_vocab)

src = torch.randint(0, src_vocab, (2, 5))  # (batch, seq_len) token ids
tgt = torch.randint(0, tgt_vocab, (2, 5))

out = model(src, tgt)
print(out.shape)  # should be (2, 5, 1000)

torch.Size([2, 5, 1000])


In [16]:
total = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total:,}")

Total parameters: 45,675,496


In [17]:
from collections import Counter
import re 

class Tokenizer:

    def __init__(self):
        super().__init__()

        self.word2idx = {}
        self.idx2word = {}
        self.special_tokens = ['<PAD>', '<UNK>', '<SOS>', '<EOS>'] 

        for idx, token in enumerate(self.special_tokens):
            self.word2idx[token] = idx
            self.idx2word[idx] = token

    def tokenize(self, text):

        text = text.lower()
        text = re.sub(r"[^a-zA-Z0-9\s]", "", text)
        tokens = text.split()
        return tokens
    
    def build_vocab(self,sentences):

        counter = Counter()

        for sentence in sentences:
            tokens = self.tokenize(sentence)
            counter.update(tokens)

        idx = len(self.special_tokens)

        for word in counter:
            if word not in self.word2idx:
                self.word2idx[word] = idx
                self.idx2word[idx] = word
                idx += 1

    def encode(self, sentence):

        tokens = self.tokenize(sentence)
        encoded_sentence = [self.word2idx.get(token, self.word2idx['<UNK>']) for token in tokens]
        return encoded_sentence
    
    def decode(self, encoded_sentence):

        decoded_sentence = [self.idx2word.get(idx, '<UNK>') for idx in encoded_sentence]
        return ' '.join(decoded_sentence)

In [18]:
from datasets import load_dataset
import torch
from torch.utils.data import Dataset

dataset = load_dataset("bentrevett/multi30k")
# English to French translation dataset

class TranslationDataset(Dataset):
    def __init__(self,split,src_tokenizer,tgt_tokenizer):
        super().__init__()
        self.dataset = dataset[split]
        self.src_tokenizer = src_tokenizer
        self.tgt_tokenizer = tgt_tokenizer

        if split == "train":
            src_sentences = [item['en'] for item in self.dataset]
            tgt_sentences = [item['de'] for item in self.dataset]
            src_tokenizer.build_vocab(src_sentences)
            tgt_tokenizer.build_vocab(tgt_sentences)

    def __len__(self):
        return len(self.dataset)
    
    def __getitem__(self,idx):
        
        item = self.dataset[idx]

        english_sentence = item['en']
        french_sentence = item['de']

        src_ids = self.src_tokenizer.encode(english_sentence)
        tgt_ids = self.tgt_tokenizer.encode(french_sentence)

        # add SOS and EOS to src too
        src_ids = [self.src_tokenizer.word2idx['<SOS>']] + src_ids + [self.src_tokenizer.word2idx['<EOS>']]

        decoder_input = [self.tgt_tokenizer.word2idx['<SOS>']] + tgt_ids
        decoder_target = tgt_ids + [self.tgt_tokenizer.word2idx['<EOS>']]

        return (
            torch.tensor(src_ids,dtype=torch.long), 
            torch.tensor(decoder_input,dtype=torch.long), 
            torch.tensor(decoder_target,dtype=torch.long)
        )
    

def collate_fn(batch):

    src_batch, decoder_input_batch, decoder_target_batch = zip(*batch)

    src_batch = torch.nn.utils.rnn.pad_sequence(
        src_batch, 
        batch_first=True, 
        padding_value=0
    )

    decoder_input_batch = torch.nn.utils.rnn.pad_sequence(
        decoder_input_batch, 
        batch_first=True, 
        padding_value=0
    )

    decoder_target_batch = torch.nn.utils.rnn.pad_sequence(
        decoder_target_batch, 
        batch_first=True, 
        padding_value=0
    )

    return src_batch, decoder_input_batch, decoder_target_batch

In [20]:
from torch.utils.data import DataLoader
#  build tokenizers
src_tokenizer = Tokenizer()
tgt_tokenizer = Tokenizer()

# load datasets
train_dataset = TranslationDataset('train', src_tokenizer, tgt_tokenizer)
val_dataset   = TranslationDataset('validation', src_tokenizer, tgt_tokenizer)

# build dataloaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, collate_fn=collate_fn)
val_loader   = DataLoader(val_dataset, batch_size=32, shuffle=False, collate_fn=collate_fn)

# check vocab sizes
print(f"src vocab size: {len(src_tokenizer.word2idx)}")
print(f"tgt vocab size: {len(tgt_tokenizer.word2idx)}")

src vocab size: 10205
tgt vocab size: 18445


In [21]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"using: {device}")

model = Transformer(
    src_vocab=10205,
    tgt_vocab=18445,
    d_model=512,
    num_heads=8,
    num_layers=6,
    d_ff=512,
    dropout=0.1
).to(device)

# loss function — ignore padding index 0
criterion = nn.CrossEntropyLoss(ignore_index=0)

# optimizer
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)

total = sum(p.numel() for p in model.parameters())
print(f"total parameters: {total:,}")

using: cpu
total parameters: 49,376,781


In [22]:
from tqdm import tqdm

def train(model, dataloader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    
    loop = tqdm(dataloader, desc="Training", leave=True)
    
    for batch_idx, (src, decoder_input, decoder_target) in enumerate(loop):
        
        src            = src.to(device)
        decoder_input  = decoder_input.to(device)
        decoder_target = decoder_target.to(device)
        
        output = model(src, decoder_input)
        
        output         = output.view(-1, output.shape[-1])
        decoder_target = decoder_target.view(-1)
        
        loss = criterion(output, decoder_target)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        
        # update tqdm bar with current loss
        loop.set_postfix(loss=f"{loss.item():.4f}")
    
    return total_loss / len(dataloader)

In [23]:
def evaluate(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0
    
    loop = tqdm(dataloader, desc="Evaluating", leave=True)
    
    with torch.no_grad():
        for src, decoder_input, decoder_target in loop:
            
            src            = src.to(device)
            decoder_input  = decoder_input.to(device)
            decoder_target = decoder_target.to(device)
            
            output = model(src, decoder_input)
            
            output         = output.view(-1, output.shape[-1])
            decoder_target = decoder_target.view(-1)
            
            loss = criterion(output, decoder_target)
            total_loss += loss.item()
            
            loop.set_postfix(loss=f"{loss.item():.4f}")
    
    return total_loss / len(dataloader)

In [24]:
num_epochs = 10

for epoch in range(num_epochs):
    train_loss = train(model, train_loader, optimizer, criterion, device)
    val_loss   = evaluate(model, val_loader, criterion, device)
    
    print(f"epoch {epoch+1}/{num_epochs} → train loss: {train_loss:.4f} | val loss: {val_loss:.4f}")

Training:   0%|          | 3/907 [00:05<28:21,  1.88s/it, loss=8.7553]


KeyboardInterrupt: 